In [1]:
import keras
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt


In [2]:
DATA_DIR = "mu3e_trigger_data"
SIGNAL_DATA_FILE = f"{DATA_DIR}/run42_sig_positions.npy"
BACKGROUND_DATA_FILE = f"{DATA_DIR}/run42_bg_positions.npy"

max_barrel_radius = 86.3
max_endcap_distance = 372.6


In [3]:
signal_data = np.load(SIGNAL_DATA_FILE)
background_data = np.load(BACKGROUND_DATA_FILE)

background_data[background_data[:, :, 0] != -1, 0] = (
    background_data[background_data[:, :, 0] != -1, 0] + max_barrel_radius
) / max_barrel_radius
background_data[background_data[:, :, 0] != -1, 1] = (
    background_data[background_data[:, :, 0] != -1, 1] + max_barrel_radius
) / max_barrel_radius
background_data[background_data[:, :, 0] != -1, 2] = (
    background_data[background_data[:, :, 0] != -1, 2] + max_endcap_distance / 2
) / max_endcap_distance

signal_data[signal_data[:, :, 0] != -1, 0] = (
    signal_data[signal_data[:, :, 0] != -1, 0] + max_barrel_radius
) / max_barrel_radius
signal_data[signal_data[:, :, 0] != -1, 1] = (
    signal_data[signal_data[:, :, 0] != -1, 1] + max_barrel_radius
) / max_barrel_radius
signal_data[signal_data[:, :, 0] != -1, 2] = (
    signal_data[signal_data[:, :, 0] != -1, 2] + max_endcap_distance / 2
) / max_endcap_distance


In [4]:
# Transfrom data to cyclic coordinates
r_bg = np.sqrt(background_data[:, :, 0] ** 2 + background_data[:, :, 1] ** 2)
phi_bg = np.arctan2(background_data[:, :, 1], background_data[:, :, 0])
z_bg = background_data[:, :, 2]
background_data_cyclindric = np.stack([r_bg, phi_bg, z_bg], axis=-1)
background_data_cyclindric[background_data[:, :, 0] == -1, :] = -1

In [5]:
class MMDLoss(keras.losses.Loss):
    def __init__(self, latent_dim, kernel='rbf', sigma=1.0, weight=1.0, **kwargs):
        super().__init__(**kwargs)
        self.latent_dim = latent_dim
        self.kernel = kernel
        self.sigma = sigma
        self.weight = weight

    def call(self, y_true, y_pred):
        z = y_pred  # shape: (batch_size, latent_dim)
        batch_size = tf.shape(z)[0]
        prior = tf.random.normal(shape=(batch_size, self.latent_dim))  # standard Gaussian

        return self.weight * self._mmd(z, prior)

    def _mmd(self, x, y):
        xx = self._compute_kernel(x, x)
        yy = self._compute_kernel(y, y)
        xy = self._compute_kernel(x, y)
        return tf.reduce_mean(xx + yy - 2 * xy)

    def _compute_kernel(self, x, y):
        x = tf.expand_dims(x, 1)  # shape: (batch, 1, dim)
        y = tf.expand_dims(y, 0)  # shape: (1, batch, dim)
        dist = tf.reduce_sum((x - y) ** 2, axis=2)
        return tf.exp(-dist / (2 * self.sigma ** 2))


class SlicedWassersteinLoss(tf.keras.losses.Loss):
    def __init__(self, latent_dim, num_projections=100, weight=1.0, **kwargs):
        super().__init__(**kwargs)
        self.latent_dim = latent_dim
        self.num_projections = num_projections
        self.weight = weight

    def call(self, y_true, y_pred):
        z = y_pred  # shape: (batch_size, latent_dim)
        batch_size = tf.shape(z)[0]

        proj_vectors = tf.random.normal((self.num_projections, self.latent_dim))
        proj_vectors = tf.math.l2_normalize(proj_vectors, axis=1)  # shape: (num_proj, latent_dim)

        z_proj = tf.matmul(z, proj_vectors, transpose_b=True)  # shape: (batch, num_proj)
        prior = tf.random.normal((batch_size, self.latent_dim))
        prior_proj = tf.matmul(prior, proj_vectors, transpose_b=True)

        z_sorted = tf.sort(z_proj, axis=0)
        prior_sorted = tf.sort(prior_proj, axis=0)

        return self.weight * tf.reduce_mean(tf.square(z_sorted - prior_sorted))

class UnitHyperSphereCoverLoss(tf.keras.losses.Loss):
    def __init__(self, latent_dim, temperature, **kwargs):
        super().__init__(**kwargs)
        self.latent_dim = latent_dim
        self.temperature = temperature

    def call(self, y_true, y_pred):
        z = y_pred  # shape: (batch_size, latent_dim)
        batch_size = tf.shape(z)[0]
        z = tf.math.l2_normalize(z, axis=1)

        # Compute pairwise squared distances
        # Using ||x - y||^2 = ||x||^2 + ||y||^2 - 2 * x^T y
        similarity_matrix = tf.matmul(z, z, transpose_b=True)
        sq_dists = 2.0 - 2.0 * similarity_matrix  # since ||z|| = 1

        # Remove diagonal (self-pairs)
        mask = tf.ones_like(sq_dists) - tf.eye(batch_size)
        sq_dists_no_diag = sq_dists * mask        # Compute the loss as the mean of the distances
        exp_dists = tf.exp(-self.temperature * sq_dists_no_diag)
        mean_exp = tf.reduce_sum(exp_dists) / tf.cast(batch_size * (batch_size - 1), tf.float32)
        return tf.math.log(mean_exp + 1e-6) / self.temperature
    

In [15]:
class EncodingLoss(keras.losses.Loss):
    def __init__(self, latent_dim, diversity_encouragement=1, name=None):
        super().__init__(name)
        self.diversity_encouragement = diversity_encouragement
        self.mmd_loss = UnitHyperSphereCoverLoss(latent_dim=latent_dim, temperature=1.0)

    def call(self, y_true, y_pred):
        latent_dim = y_pred.shape[-1] // 2
        input, ae_output = tf.split(y_pred, [latent_dim, latent_dim], axis=-1)
        ae_loss = tf.reduce_mean(tf.square(input - ae_output))

        # include a regularization term to encourage diversity in the latent space to encourage the model to learn a variance of 1
        if self.diversity_encouragement > 0:
            diversity_loss = tf.reduce_sum(
                tf.square(
                    tf.ones(
                        input.shape[-1],
                    )
                    - tf.math.reduce_variance(input, axis=0)
                )
            )
        else:
            diversity_loss = 0.0

        
        return ae_loss + self.mmd_loss(input, input) + self.diversity_encouragement * diversity_loss


class ReconstructionQuality(keras.metrics.Metric):
    def __init__(self, name="reconstruction_quality", **kwargs):
        super().__init__(name=name, **kwargs)
        self.total_loss = self.add_weight(name="total_loss", initializer="zeros")
        self.count = self.add_weight(name="count", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        latent_dim = y_pred.shape[-1] // 2
        input, ae_output = tf.split(y_pred, [latent_dim, latent_dim], axis=-1)
        loss = tf.reduce_mean(tf.square(input - ae_output))
        self.total_loss.assign_add(loss)
        self.count.assign_add(1)

    def result(self):
        return self.total_loss / self.count


class FeatureVariance(keras.metrics.Metric):
    def __init__(self, name="feature_variance", **kwargs):
        super().__init__(name=name, **kwargs)
        self.variance = self.add_weight(name="variance", initializer="zeros")
        self.count = self.add_weight(name="count", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        latent_dim = y_pred.shape[-1] // 2
        input, ae_output = tf.split(y_pred, [latent_dim, latent_dim], axis=-1)
        variance = tf.math.reduce_variance(input, axis=0)
        self.variance.assign_add(tf.reduce_mean(variance))
        self.count.assign_add(1)

    def result(self):
        return self.variance / self.count

In [16]:
sequence_length = 256
feature_dim = 3
hidden_dim = 6
latent_dim = 32

In [ ]:
from src.model.components import (
    SelfAttentionStack,
    MLP,
    PoolingAttentionBlock,
    PointTransformerFromCoords,
    DecoderQueries,
    MultiHeadAttentionBlock,
)
from src.model.components import (
    GenerateDecoderMask,
    GenerateMask,
    MaskOutput,
    GetSequenceLength,
)

# Fixed size encoding models
input = keras.Input(shape=(sequence_length, feature_dim), name="input")
mask = GenerateMask(name="generate_mask")(input)
sequence_length_layer = GetSequenceLength(name="get_sequence_length")(mask)

input_embedding = PointTransformerFromCoords(
    name="point_transformer_from_coords",
    feature_dim=hidden_dim,
    pos_mlp_hidden_dim=hidden_dim,
    attn_mlp_hidden_dim=hidden_dim,
)(input, mask)

attention_block = SelfAttentionStack(
    name="self_attention_block", num_heads=4, key_dim=hidden_dim, stack_size=2
)(input_embedding, mask)
#pooling = keras.layers.GlobalAveragePooling1D(name="global_average_pooling")(attention_block, mask)

attention_pooling = PoolingAttentionBlock(
    name="pooling_attention_block",
    num_heads=8,
    key_dim=hidden_dim,
    dropout_rate=0.0,
    num_seed_vectors=hidden_dim,
)(attention_block, mask)
fixed_size_encoding = MLP(name="fixed_size_encoding", output_dim=1, num_layers=3)(attention_block)
fixed_size_encoding = keras.layers.Flatten(name="flatten")(attention_pooling)

#fixed_size_encoding = MLP(name="mlp", output_dim=1)(fixed_size_encoding)

# Autoencoder model
encoder = MLP(name="encoder", output_dim=int(latent_dim / 1.5), num_layers=4)
decoder = MLP(name="decoder", output_dim=latent_dim, num_layers=4)


output = keras.layers.Concatenate(name="concatenate")([fixed_size_encoding, decoder(encoder(fixed_size_encoding))])

# Define model
autoencoder_layers = [encoder, decoder]
# A: Embedding Model (input to fixed-size embedding)
embedding_model = keras.Model(inputs=input, outputs=fixed_size_encoding, name="embedding_model")

# B: Autoencoder Model (fixed-size embedding to reconstructed latent vector)
autoencoder_input = keras.Input(shape=(fixed_size_encoding.shape[-1],), name="ae_input")
encoded = encoder(autoencoder_input)
decoded = decoder(encoded)
ae_output = keras.layers.Concatenate(name="ae_concat")([autoencoder_input, decoded])
autoencoder_model = keras.Model(inputs=autoencoder_input, outputs=ae_output, name="autoencoder_model")

# C: Full Model (input to concatenated fixed + decoded vector)
full_model = keras.Model(inputs=input, outputs=output, name="full_model")

In [18]:
loss_fn = EncodingLoss(diversity_encouragement=0.2, latent_dim = latent_dim)
metrics = [ReconstructionQuality(), FeatureVariance()]

full_model.compile(optimizer=keras.optimizers.Adam(1e-4), loss=loss_fn, metrics=metrics)
autoencoder_model.compile(optimizer=keras.optimizers.Adam(1e-2), loss=EncodingLoss(diversity_encouragement = 0, latent_dim = latent_dim))


In [ ]:
x_train = background_data[:20000]
epochs = 20
autoencoder_train_every = 2
autoencoder_train_steps = 10
batch_size = 1024

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    if epoch % autoencoder_train_every == 1:
        print("➡️ Training only autoencoder")

        # Step 1: Freeze input embedding and extract encodings
        fixed_embeddings = embedding_model.predict(x_train, batch_size=batch_size, verbose=0)

        autoencoder_model.fit(
            x=fixed_embeddings,
            y=fixed_embeddings,
            batch_size=batch_size,
            epochs=autoencoder_train_steps,
            verbose=1
        )
    else:
        print("🔁 Training full model")
        full_model.fit(
            x=x_train,
            y=x_train,
            batch_size=batch_size,
            epochs=1,
            verbose=1
        )



Epoch 1/20
🔁 Training full model
20/20 ━━━━━━━━━━━━━━━━━━━━ 224s 11s/step - feature_variance: 0.0296 - loss: 7.3146 - reconstruction_quality: 0.9527

Epoch 2/20
➡️ Training only autoencoder

Epoch 3/20
🔁 Training full model
20/20 ━━━━━━━━━━━━━━━━━━━━ 253s 13s/step - feature_variance: 0.0614 - loss: 6.4064 - reconstruction_quality: 0.5074

Epoch 4/20
🔁 Training full model
20/20 ━━━━━━━━━━━━━━━━━━━━ 217s 11s/step - feature_variance: 0.0987 - loss: 5.8622 - reconstruction_quality: 0.4827

Epoch 5/20
➡️ Training only autoencoder

Epoch 6/20
🔁 Training full model
20/20 ━━━━━━━━━━━━━━━━━━━━ 236s 12s/step - feature_variance: 0.2130 - loss: 4.5163 - reconstruction_quality: 0.5242

Epoch 7/20
🔁 Training full model
20/20 ━━━━━━━━━━━━━━━━━━━━ 255s 13s/step - feature_variance: 0.5192 - loss: 2.2026 - reconstruction_quality: 0.7668

Epoch 8/20
➡️ Training only autoencoder

Epoch 9/20
🔁 Training full model
20/20 ━━━━━━━━━━━━━━━━━━━━ 269s 13s/step - feature_variance: 0.7333 - loss: 1.0492 - reconstr

In [14]:
embedding_model.predict(x_train[:10])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


array([[ 0.2030942 ,  0.2693734 , -1.4560745 ,  1.8446772 , -0.4687941 ,
        -0.38766542,  0.2694225 ,  0.49200127, -1.868186  ,  1.4191997 ,
        -0.4074923 ,  0.10048641,  0.46678823,  0.38057283, -1.6325397 ,
         1.61539   , -0.51809084, -0.30727383,  0.07617234,  0.43888852,
        -1.8126392 ,  1.5393155 , -0.35968676,  0.12337501,  0.20002136,
         0.6338173 , -1.8094099 ,  1.4550463 , -0.39982304, -0.07422651,
         0.01251092, -0.11751951, -1.085353  ,  2.0957267 , -0.3458848 ,
        -0.55565876],
       [ 0.2568196 ,  0.38833737, -1.7443695 ,  1.5958726 , -0.43531325,
        -0.0561513 ,  0.20671505,  0.5102046 , -1.9257858 ,  1.2618855 ,
        -0.4643924 ,  0.41674495,  0.5289902 ,  0.50996417, -1.8573102 ,
         1.3210893 , -0.50827456,  0.0107563 ,  0.05718765,  0.47838545,
        -1.899377  ,  1.3699785 , -0.37362903,  0.3729422 ,  0.22167432,
         0.7506418 , -1.9594343 ,  1.1498176 , -0.39963245,  0.2425158 ,
         0.0652882 , -0.00605

In [ ]:
fixed_embeddings